# v36 — TransReID / DeiT-Small: ночной поиск качества

**Run All**: проверка → 24 конфигурации × 400 шагов → 4 финалиста до 1800 шагов → таблица и сохранённые лучшие checkpoint. Ограничения по времени нет: очередь завершается после всех заданных опытов, даже если потребуется больше одной ночи.

Сетка: global / JPM × learning rate 1e-5 / 3e-5 / 1e-4 × weight decay 1e-4 / 1e-2 × soft-triplet / SupCon. Batch — **16 фотографий (8 ID × 2)**. Шаги — обновления весов, не эпохи и не блоки по 200 фотографий. Всего 15 200 обновлений при успешном прохождении всех опытов.

Обучение — только свои train-ID. Сравнение — на прежнем внутреннем holdout, с исходным R1 на том же разбиении. Для каждой модели считаются raw и less_graph, отдельно и в фиксированной смеси 90% R1 / 10% TransReID. **v25, исходная validation, bbox и порог не меняются. NiVe не используется для этого обучения.**


In [1]:
import os
os.environ['ORT_DISABLE_TELEMETRY'] = '1'  # до импорта runtime

from pathlib import Path
import sys
import subprocess
import time

DEVICE = 'mps'       # Apple Silicon; 'cuda' только на доступном NVIDIA GPU
RUN_NAME = 'night_v1' # то же имя = проверенное продолжение, новое имя = новый прогон

candidates = [p for root in [Path.cwd(), *Path.cwd().parents]
              for p in [root, root / 'Car-classification-MSK']]
ROOT = next((p for p in candidates if (p / 'training/transreid_night.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри каталога проекта Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
started = time.perf_counter()
print('Python:', sys.executable)
print('Project:', ROOT)
print('Device:', DEVICE, '| Run:', RUN_NAME)


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Project: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Device: mps | Run: night_v1


## 1. Публичные веса и устройство

Веса уже загружены локально. Если файл отсутствует на другом компьютере, эта ячейка скачает его с официального адреса Facebook и проверит SHA256. Доступ к gated-моделям и токен не нужны. После подготовки инференс/обучение не требуют сети. Не включайте скрытый MPS→CPU fallback.


In [2]:
from training import transreid_model as vision, transreid_night as night
print('Device checked:', vision.device_for(DEVICE))
print('Verified public weights:', vision.download_weights())
print('Weight SHA256:', vision.WEIGHT_SHA256)


Device checked: mps
Verified public weights: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_36_transreid_night/weights/deit_small_distilled_patch16_224-649709d9.pth
Weight SHA256: 649709d94f9fd790ea86c16f99d788e709b86a1f64315a19d887895f9948fb09


## 2. Короткие технические проверки

Проверяются обе архитектуры, обе функции потерь, сохранение/продолжение, независимость запросов и сама очередь на маленьких синтетических данных. Это не оценка качества обученной модели. При ошибке смотрите вывод тестов над traceback; не пропускайте ячейку. Полное обучение начнётся только ниже.


In [3]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_transreid_night.py',
                'tests/test_retrieval_policy.py',
                'tests/test_dual_role_inference.py',
                'tests/test_nive_system.py'], check=True, cwd=ROOT,
               env={**os.environ, 'ORT_DISABLE_TELEMETRY': '1'})


........................................................................ [ 63%]
.........................................                                [100%]
113 passed in 17.15s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_transreid_night.py', 'tests/test_retrieval_policy.py', 'tests/test_dual_role_inference.py', 'tests/test_nive_system.py'], returncode=0)

## 3. Проверка данных и фиксация серии

Нужны примерно **13 GiB свободного диска** для нового полного прогона, включая запас. Программа ничего не удаляет из старых опытов. Внутри новой серии заменяется только устаревшая служебная копия optimizer-resume; все selectable checkpoint сохраняются. При продолжении учитывается уже занятое серией место.

Разбиение, порядок изображений, исходные данные, веса, код, гиперпараметры и среда фиксируются. Если они изменились, старый RUN_NAME не используется для другого эксперимента. Не меняйте код/конфигурацию во время работы. Подключите питание и не давайте компьютеру заснуть.


In [4]:
context = night.prepare(run_name=RUN_NAME, device=DEVICE)
import pandas as pd
from IPython.display import display, Markdown
display(pd.DataFrame(context['trials']))
print('Train images:', len(context['target']), '| evaluation images:', len(context['rows']))
print('Results:', context['output'])


PREFLIGHT: 24 trials, 740 train-ID / 185 holdout-ID, 1036 evaluation images; outer disabled


,id,architecture,encoder_lr,weight_decay,metric_loss
0,T01_global_soft_triplet,global,0.00001,0.0001,soft_triplet
1,T02_global_supcon,global,0.00001,0.0001,supcon
2,T03_global_soft_triplet,global,0.00001,0.0100,soft_triplet
3,T04_global_supcon,global,0.00001,0.0100,supcon
4,T05_global_soft_triplet,global,0.00003,0.0001,soft_triplet
5,T06_global_supcon,global,0.00003,0.0001,supcon
6,T07_global_soft_triplet,global,0.00003,0.0100,soft_triplet
7,T08_global_supcon,global,0.00003,0.0100,supcon
8,T09_global_soft_triplet,global,0.00010,0.0001,soft_triplet
9,T10_global_supcon,global,0.00010,0.0001,supcon


Train images: 4612 | evaluation images: 1036
Results: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_36_transreid_night/runs/night_v1


## 4. Проверка на выбранном устройстве

Четыре пробных обновления на настоящем batch 16 проверяют операции и память global/JPM с обеими функциями потерь. Эти временные модели не участвуют в сравнении. Печатаемое время — короткая проверка с холодным запуском, не точный прогноз длительности ночи и не конкурсный benchmark.


In [5]:
with night.base.old.run_lock(context['output']):
    smoke = night.runtime_smoke(context)
display(pd.DataFrame(smoke['updates']))


STAGE runtime_smoke: start
using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
Resized position embedding from size:torch.Size([1, 197, 384]) to size: torch.Size([1, 257, 384]) with height:16 width: 16
DEVICE SMOKE global/soft_triplet: batch 16, loss=8.3240, 0.67s
DEVICE SMOKE global/supcon: batch 16, loss=8.4172, 0.28s
using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
Resized position embedding from size:torch.Size([1, 197, 384]) to size: torch.Size([1, 257, 384]) with height:16 width: 16
DEVICE SMOKE jpm/soft_triplet: batch 16, loss=8.7626, 0.39s
DEVICE SMOKE jpm/supcon: batch 16, loss=8.3422, 0.30s
STAGE runtime_smoke: complete in 2.9s


,architecture,metric,loss,seconds
0,global,soft_triplet,8.323975,0.674331
1,global,supcon,8.417159,0.275594
2,jpm,soft_triplet,8.762606,0.385379
3,jpm,supcon,8.342233,0.300846


## 5. Вся очередь — без ручного выбора между опытами

Сначала две модели без дообучения. Затем все 24 обучения до 400 шагов. По каждой архитектуре продолжаются лучшие по самостоятельному mAP и по смеси с R1; если это один и тот же опыт, второе место получает следующий по смеси. Финалисты продолжаются с тем же optimizer и расписанием learning rate до 1800 шагов, без перезапуска.

Checkpoint качества: 400 / 800 / 1200 / 1800. Прогресс обучения: каждые 25 шагов. Resume: каждые 50. Метрики печатаются после извлечения признаков; итоговый отчёт обновляется по мере продвижения. NaN/нехватка памяти отдельного опыта отмечаются как неуспех, остальные продолжаются **без автоматического уменьшения batch или изменения рецепта**. Неизвестные ошибки, повреждение данных и протокола останавливают выполнение.

После обычного прерывания: перезапустите kernel и Run All с тем же RUN_NAME. Уже проверенные результаты не считаются заново и не выдаются за новые измерения. Файл .lock остаётся штатно; при завершении процесса операционная система освобождает блокировку. Удалять его вручную не нужно.


In [ ]:
results = night.run(context)


## 6. Результат

Сохраняются лучший самостоятельный encoder и лучший вариант смеси, а не только усреднённый рецепт. Таблица относится к одному многократно использованному внутреннему holdout; три draw пересекаются и не являются независимыми тестами. Отбор после 400 шагов может пропустить медленно сходящийся вариант.

Рост относительно fold-matched R1 — повод для следующего системного сравнения, а не автоматическая победа над full-train v25. Ни порог, ни приложение, ни исходная validation этим notebook не изменяются.


In [ ]:
display(Markdown((context['output'] / 'REPORT.md').read_text(encoding='utf-8')))
print('Best single:', results['best_single'])
print('Best fixed mixture:', results['best_mixture'])
print('Delta mixture vs fold-matched R1:', results['mixture_delta_to_r1'])
print('Failed configurations:', results['failures'])
print('This notebook session elapsed:', night.base.format_duration(time.perf_counter()-started))
print('Saved report:', context['output'] / 'REPORT.md')
